In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [7]:
df = pd.read_csv('titanic.csv')
df = df.fillna({'Cabin': 'NAN'})  # Fill missing values in 'Cabin' column with 'NAN'
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NAN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NAN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NAN,S


In [8]:
categorical_columns = ['Sex', 'Embarked', 'Cabin']
def encode_categorical_columns(df, categorical_columns):
    df = df.copy()
    for column in categorical_columns:
        le = LabelEncoder()
        column_name = column + '_encoded'
        df[column_name] = le.fit_transform(df[column])
    return df

df_encoded = encode_categorical_columns(df, categorical_columns)
df_encoded.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Sex_encoded,Embarked_encoded,Cabin_encoded
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NAN,S,1,2,146
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0,0,81
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NAN,S,0,2,146
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,0,2,55
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NAN,S,1,2,146


In [9]:
feature_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded', 'Embarked_encoded', 'Cabin_encoded']
target_column = 'Survived'
def decision_tree_pipeline(df, feature_columns, target_column):
    X = df[feature_columns]
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    confusion_matrix = pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted']).rename(columns={0: 'Not Survived', 1: 'Survived'}, index={0: 'Not Survived', 1: 'Survived'})
    print('Confusion Matrix:\n', confusion_matrix)
    print(f'Decision Tree Classifier Accuracy: {accuracy:.4f}')
    return model, scaler

decision_tree_model, scaler = decision_tree_pipeline(df_encoded, feature_columns, target_column)


Confusion Matrix:
 Predicted     Not Survived  Survived
Actual                              
Not Survived            92        13
Survived                19        55
Decision Tree Classifier Accuracy: 0.8212
